# CatBoost Meta-Learner on Kaggle

Clones `approach/catboost-meta`. Replaces `combined_agent.py`'s hand-tuned
blend (candidate-trust formula + 30/70 ngram/neural split) with a learned
combination: a CatBoost classifier trained on real synthetic game states
to predict "is this letter actually in the word" from the three raw
signal scores (candidate entropy, n-gram, BiLSTM) plus a few auxiliary
features.

This exists because local testing hit a real environment wall: torch and
sklearn cannot coexist in the same Python process on that machine
(confirmed both import orders -- one segfaults, one fails DLL init).
Kaggle doesn't have this problem, and has CatBoost preinstalled, which is
also what the reference approach (github.com/Aditya-dom/Auto_Hangman,
reporting 95% in-dict / 67% out-of-dict) actually used.

**Before running:** in the notebook's Settings panel (right sidebar), set
**Accelerator = GPU T4 x2** (or any GPU, for the BiLSTM half) and
**Internet = On**.

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
import catboost
print('catboost', catboost.__version__)

In [ ]:
REPO_URL = "https://github.com/Sahoo-Achyutananda/MELTWATER---HACKATHON.git"
BRANCH = "approach/catboost-meta"

!rm -rf repo
!git clone --branch $BRANCH --single-branch $REPO_URL repo
%cd repo/brand-buzzword-hackathon
!ls

## Train the BiLSTM half first

The neural signal is one of CatBoost's input features, so this has to
exist before generating the meta-learner's training data.

In [ ]:
!python src/train_bilstm.py --epochs 40

## Generate the meta-learner training data + train CatBoost

40,000 synthetic game states (random word, random mask fraction, random
synthetic wrong-guesses -- same recipe as the BiLSTM's own MLM training),
each contributing one row per still-unguessed letter. Prints feature
importances at the end -- worth checking whether CatBoost actually leans
on all three signals or mostly ignores one.

In [ ]:
!python src/train_catboost_meta.py --n-states 40000

## Validate

Same held-out-train.txt methodology as every other branch. The number
that matters: does the learned combination beat approach/full-combo-
conv1d's hand-tuned blend (entropy-based candidate scoring, ~53.8-54% on
the full held-out set)? Use --full for the reliable 22,530-word number
instead of the noisier 3000-word sample.

In [ ]:
!python src/validate_catboost_combined.py --full

## Generate submission.csv

Only run this after confirming the validation number actually beats
full-combo-conv1d -- otherwise this is a slow way to find out it doesn't.

In [ ]:
!python src/generate_submission_catboost_combined.py

## Save outputs

In [ ]:
import shutil
shutil.copy("src/bilstm_conv_attn_feat_masker.pt", "/kaggle/working/bilstm_conv_attn_feat_masker.pt")
shutil.copy("src/catboost_meta.cbm", "/kaggle/working/catboost_meta.cbm")
shutil.copy("submission.csv", "/kaggle/working/submission.csv")
print("saved checkpoints + submission.csv to /kaggle/working/ -- download from the Output tab")